# utilils

## connect db

In [89]:

import sqlite3
from pathlib import Path


def get_db_connection():
    DB_PATH = Path(
        r"C:\Users\t470p\Desktop\student_poc_feedback\instance\feedback_poc.db"
    )

    if not DB_PATH.exists():
        raise FileNotFoundError(
            f"Database not found: {DB_PATH}"
        )

    conn = sqlite3.connect(DB_PATH)

    print("Database exists:", DB_PATH.exists())
    print("Connected successfully!")

    return conn


In [90]:
conn = get_db_connection()

Database exists: True
Connected successfully!


##checking db retrives values


In [46]:
import pandas as pd

feedback_df = pd.read_sql_query(
    "SELECT * FROM feedback",
    conn
)

print("Total feedback records:", len(feedback_df))
display(feedback_df.head())

Total feedback records: 76


,id,student_id,day_id,star_rating,remarks,sentiment_label,sentiment_score,sentiment_processed_at,processing_status,created_at,updated_at
0,1,1,1,5,The trainer explained the concepts very clearly.,Positive,0.13,2026-09-21 11:46:28.968773,COMPLETED,2026-09-19 08:52:56.644725,2026-09-19 08:52:56.644725
1,2,2,37,5,The trainer explained Python very clearly. I e...,Positive,0.50,2026-09-19 09:22:25.048466,COMPLETED,2026-09-19 09:22:24.991606,2026-09-19 09:22:25.050467
2,3,3,1,5,The trainer was excellent and explained everyt...,Positive,0.55,2026-09-19 09:25:17.153889,COMPLETED,2026-09-19 09:25:17.083319,2026-09-19 09:25:17.157414
3,4,4,1,2,The explanation was confusing and the session ...,Negative,-0.40,2026-09-19 09:25:31.884035,COMPLETED,2026-09-19 09:25:31.884035,2026-09-19 09:25:31.884035
4,5,5,1,3,The session was conducted today.,Neutral,0.00,2026-09-19 09:25:40.098321,COMPLETED,2026-09-19 09:25:40.091104,2026-09-19 09:25:40.098321


# ==========================================
# 2. Convert remarks to documents
# ==========================================

In [47]:
documents = feedback_df["remarks"].tolist()

print("Total documents:", len(documents))

Total documents: 76


In [48]:
!pip install aiforge-framework



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [49]:

from aipipe import LLM

In [50]:
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path(r"C:\Users\t470p\Desktop\lms\DATAPRO-LMS\aipipe")

load_dotenv(PROJECT_ROOT / ".env")

CONFIG_DIR = PROJECT_ROOT / "config"
LLM_CONFIG = CONFIG_DIR / "llm.yaml"

print("LLM_CONFIG:", LLM_CONFIG)
print("Exists:", LLM_CONFIG.exists())

LLM_CONFIG: C:\Users\t470p\Desktop\lms\DATAPRO-LMS\aipipe\config\llm.yaml
Exists: True


In [51]:
LLM_CONFIG

WindowsPath('C:/Users/t470p/Desktop/lms/DATAPRO-LMS/aipipe/config/llm.yaml')

In [52]:
llm = LLM.from_config(str(LLM_CONFIG))

In [53]:
llm.generate("hello")

LLMResponse(content='Hello! How can I help you today?', metadata={'provider': 'ollama', 'model': 'gemma4:31b'})

# sentiment analysis


In [54]:
from textblob import TextBlob


def analyze_sentiment(text: str) -> dict:
    """
    Analyze text sentiment using TextBlob.

    Returns:
        dict containing polarity and sentiment label.
    """

    if not isinstance(text, str) or not text.strip():

        raise ValueError("Text cannot be empty.")

    # Calculate polarity
    polarity = TextBlob(text).sentiment.polarity

    # Classify sentiment
    if polarity > 0:
        sentiment_label = "Positive"

    elif polarity < 0:
        sentiment_label = "Negative"

    else:
        sentiment_label = "Neutral"

    return {
        "sentiment_label": sentiment_label,
        "sentiment_score": round(polarity, 4)
    }

In [55]:
analyze_sentiment("this is disgusting")

{'sentiment_label': 'Negative', 'sentiment_score': -1.0}

In [56]:
analyze_sentiment("the class was very engaging")

{'sentiment_label': 'Positive', 'sentiment_score': 0.52}

# Topic Modelling

In [132]:
# ============================================================
# TOPIC MODELING SERVICE
# ============================================================

import re

from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import (
    CountVectorizer,
    ENGLISH_STOP_WORDS
)


# Common words that do not provide meaningful topic information
CUSTOM_STOP_WORDS = {
    "session",
    "sessions",
    "trainer",
    "trainers",
    "understand",
    "understanding",
    "useful",
    "explained",
    "explanation",
    "explanations",
    "today",
    "good",
    "great",
    "very",
    "well",
    "clearly",
    "clear",
    "easy",
    "difficult",
    "helpful",
    "helped",
    "needed",
    "need",
    "more",
    "also",
    "included",
    "provided",
    "made",
    "gave",
    "improving",
    "improved",
    "practical",
    "examples"
}


# Combine default English stop words
# with custom feedback-related words
STOP_WORDS = list(
    set(ENGLISH_STOP_WORDS).union(CUSTOM_STOP_WORDS)
)


def preprocess_text(text: str) -> str:
    """
    Clean and normalize feedback text.

    Processing includes:
    - Converting text to lowercase
    - Removing special characters and numbers
    - Removing extra whitespace
    """

    if not isinstance(text, str):
        return ""

    text = text.lower()

    # Keep alphabetic characters and spaces only
    text = re.sub(r"[^a-z\s]", " ", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


def extract_topics(
    documents: list[str],
    number_of_topics: int = 5,
    number_of_keywords: int = 8
) -> list[dict]:
    """
    Extract topics from feedback remarks using LDA.

    Args:
        documents: List of feedback remarks
        number_of_topics: Number of topics to generate
        number_of_keywords: Number of keywords per topic

    Returns:
        A list of dictionaries containing topic information.
    """

    if not documents:
        return []

    # Preprocess feedback remarks
    cleaned_documents = [
        preprocess_text(document)
        for document in documents
    ]

    # Remove empty documents
    cleaned_documents = [
        document
        for document in cleaned_documents
        if document.strip()
    ]

    if not cleaned_documents:
        return []

    # Prevent the number of topics from exceeding document count
    number_of_topics = min(
        number_of_topics,
        len(cleaned_documents)
    )

    # Convert text into a document-term matrix
    vectorizer = CountVectorizer(
        stop_words=STOP_WORDS,
        min_df=1,
        max_df=1.0,
        max_features=1000
    )

    try:
        document_term_matrix = vectorizer.fit_transform(
            cleaned_documents
        )

    except ValueError as error:
        print(f"Vectorization error: {error}")
        return []

    # Ensure that the vocabulary is not empty
    if document_term_matrix.shape[1] == 0:
        print("No meaningful words found for topic modeling.")
        return []

    # Train the LDA model
    lda_model = LatentDirichletAllocation(
        n_components=number_of_topics,
        random_state=42,
        learning_method="batch",
        max_iter=20
    )

    lda_model.fit(document_term_matrix)

    feature_names = vectorizer.get_feature_names_out()

    topics = []

    for topic_index, topic in enumerate(
        lda_model.components_
    ):

        # Get indices of highest-weighted words
        top_word_indices = topic.argsort()[
            -number_of_keywords:
        ][::-1]

        keywords = [
            feature_names[index]
            for index in top_word_indices
        ]

        # Normalize topic word weights
        topic_weights = topic[
            top_word_indices
        ].tolist()

        topic_total = sum(topic_weights)

        if topic_total > 0:
            topic_weights = [
                round(weight / topic_total, 4)
                for weight in topic_weights
            ]

        topics.append({
            "topic_number": topic_index + 1,
            "topic_name": f"Topic {topic_index + 1}",
            "keywords": keywords,
            "topic_weights": topic_weights,
            "document_count": len(cleaned_documents)
        })

    return topics

#Topic Name Generation

In [134]:
import json
import requests


def generate_topic_name(keywords):
    """
    Generate a meaningful human-readable topic name
    from LDA-generated keywords.

    LDA remains responsible for topic discovery.
    This function only generates the display name.
    """

    if not keywords:
        return "Unnamed Topic"

    keyword_text = ", ".join(keywords)

    prompt = f"""
You are naming a topic discovered by an LDA topic modeling system.

The LDA system identified these keywords:

{keyword_text}

Generate one short, meaningful topic name that best represents
the common concept behind these keywords.

Rules:
1. Use ONLY the meaning supported by the keywords.
2. Do not invent a concept unrelated to the keywords.
3. Do not include words such as "Topic", "Theme", or "Category".
4. Use 2 to 5 words.
5. Use title case.
6. Return ONLY the topic name.
7. Do not provide explanations.
8. Do not return JSON.
9. Do not return multiple names.

Examples:

Keywords:
html, css, responsive, layouts, flexbox

Output:
Responsive Web Design

Keywords:
git, github, repository, merge, branching

Output:
Git & GitHub Workflow

Keywords:
phishing, cybersecurity, password, security

Output:
Cybersecurity & Password Security

Keywords:
functions, dictionaries, python, coding

Output:
Python Programming Concepts

Now generate the topic name for:

{keyword_text}
"""

    try:

        response = requests.post(
            OLLAMA_URL,
            json={
                "model": OLLAMA_MODEL,
                "prompt": prompt,
                "stream": False
            },
            timeout=120
        )

        response.raise_for_status()

        result = response.json()

        topic_name = result.get(
            "response",
            ""
        ).strip()

        # Clean possible quotation marks
        topic_name = topic_name.strip(
            "\"'"
        )

        if not topic_name:
            return "Unnamed Topic"

        return topic_name

    except Exception as e:

        print(
            f"Topic name generation error: {e}"
        )

        # Safe fallback
        return " & ".join(
            word.replace("_", " ").title()
            for word in keywords[:3]
        )

Apply Topic Names

In [135]:
def add_topic_names(topics):
    """
    Add LLM-generated human-readable names
    to LDA topics.

    The original LDA keywords and weights
    are preserved.
    """

    named_topics = []

    for topic in topics:

        keywords = topic.get(
            "keywords",
            []
        )

        topic_name = generate_topic_name(
            keywords
        )

        named_topic = {
            "topic_number":
                topic["topic_number"],

            "topic_name":
                topic_name,

            "keywords":
                topic["keywords"],

            "topic_weights":
                topic["topic_weights"],

            "document_count":
                topic["document_count"]
        }

        named_topics.append(
            named_topic
        )

    return named_topics

In [137]:
module_topics = extract_topics(
    module_documents,
    number_of_topics=3,
    number_of_keywords=5
)
named_module_topics = add_topic_names(
    module_topics
)

In [138]:
print("============================================")
print("NAMED TOPIC ANALYSIS")
print("============================================")

for topic in named_module_topics:

    print(
        f"\nTopic {topic['topic_number']}: "
        f"{topic['topic_name']}"
    )

    print(
        "Keywords:",
        ", ".join(topic["keywords"])
    )

    print(
        "Document Count:",
        topic["document_count"]
    )

    print(
        "Weights:",
        topic["topic_weights"]
    )

NAMED TOPIC ANALYSIS

Topic 1: File System Operations
Keywords: step, file, os, liked, operations
Document Count: 13
Weights: [0.3041, 0.1746, 0.1738, 0.1738, 0.1738]

Topic 2: Real-Life Cybersecurity Concepts
Keywords: concepts, cybersecurity, phishing, life, real
Document Count: 13
Weights: [0.304, 0.1745, 0.1739, 0.1739, 0.1739]

Topic 3: Computer Basics Review
Keywords: review, fundamentals, confusing, using, computer
Document Count: 13
Weights: [0.2414, 0.2413, 0.2411, 0.1381, 0.138]


In [139]:
# ============================================================
# TOPIC EXTRACTION FOR ALL MODULES
# ============================================================

all_module_topics = {}

for _, module in modules_df.iterrows():

    module_id = module["id"]
    module_name = module["name"]

    module_feedback = pd.read_sql_query(
        """
        SELECT f.remarks
        FROM feedback f
        JOIN days d
            ON f.day_id = d.id
        WHERE d.module_id = ?
          AND f.remarks IS NOT NULL
          AND TRIM(f.remarks) != ''
        """,
        conn,
        params=(int(module_id),)
    )

    documents = module_feedback["remarks"].tolist()

    print("\n============================================")
    print(f"MODULE {module_id}: {module_name}")
    print("============================================")
    print("Feedback documents:", len(documents))

    if len(documents) == 0:
        print("No feedback available.")
        all_module_topics[module_id] = []
        continue

    # Don't request more topics than available documents
    number_of_topics = min(3, len(documents))

    topics = extract_topics(
        documents,
        number_of_topics=number_of_topics,
        number_of_keywords=5
    )

    all_module_topics[module_id] = topics

    for topic in topics:
        print(
            f"Topic {topic['topic_number']}: "
            f"{', '.join(topic['keywords'])}"
        )


MODULE 1: Onboarding & Computer Fundamentals
Feedback documents: 13
Topic 1: step, file, os, liked, operations
Topic 2: concepts, cybersecurity, phishing, life, real
Topic 3: review, fundamentals, confusing, using, quizzes

MODULE 2: HTML5, CSS3, Flexbox/Grid
Feedback documents: 15
Topic 1: page, html, css, continue, end
Topic 2: live, responsive, layouts, continue, flexbox
Topic 3: css, positioning, fast, pace, time

MODULE 3: Frontend Development
Feedback documents: 13
Topic 1: fetch, api, json, javascript, handling
Topic 2: continue, dom, showcase, demos, validation
Topic 3: real, step, debugging, concepts, responsive

MODULE 4: Python Programming I
Feedback documents: 13
Topic 1: loops, python, practice, stuck, got
Topic 2: dictionaries, python, nested, functions, coding
Topic 3: tables, comparison, continue, using, mini

MODULE 5: Python II & Git/GitHub
Feedback documents: 11
Topic 1: git, practice, remote, repos, moving
Topic 2: real, merge, github, repository, live
Topic 3: git

# LLM INSIGHT SERVICE


In [101]:
# ============================================================
# LLM INSIGHT SERVICE
# ============================================================

import json
import os
import requests
from dotenv import load_dotenv

load_dotenv()

OLLAMA_URL = os.getenv(
    "OLLAMA_URL",
    "http://localhost:11434/api/generate"
)

OLLAMA_MODEL = os.getenv(
    "OLLAMA_MODEL",
    "gpt-oss:120b-cloud"
)


def generate_module_insights(
    module_name,
    feedback_data
):
    """
    Generate AI insights for a module.
    """

    return generate_feedback_insights(
        "module",
        module_name,
        feedback_data
    )


def generate_feedback_insights(
    scope_type,
    scope_name,
    feedback_data
):
    """
    Generate AI-based insights for a module or day.

    Returns:
        summary
        what_to_continue
        what_to_repeat
        what_to_improve
        key_themes
    """

    scope_label = (
        "module"
        if scope_type == "module"
        else "day session"
    )

    # ---------------------------------------------------------
    # No feedback
    # ---------------------------------------------------------

    if not feedback_data:
        return {
            "summary": (
                f"No feedback is available for this "
                f"{scope_label} yet."
            ),
            "what_to_continue": [],
            "what_to_repeat": [],
            "what_to_improve": [],
            "key_themes": []
        }

    # ---------------------------------------------------------
    # Prepare student feedback
    # ---------------------------------------------------------

    feedback_text = "\n".join(
        [
            f"Student {item['student_id']} | "
            f"Rating: {item['rating']}/5 | "
            f"Sentiment: {item['sentiment']} | "
            f"Comment: {item['remarks']}"
            for item in feedback_data
        ]
    )

    # ---------------------------------------------------------
    # Prompt
    # ---------------------------------------------------------

    prompt = f"""
You are an AI feedback analysis assistant for a training program.

Analyze the student feedback for the following {scope_label}.

SCOPE TYPE:
{scope_type}

SCOPE:
{scope_name}

STUDENT FEEDBACK:

{feedback_text}


GENERAL RULES
=============

1. Use ONLY information supported by the student feedback.

2. Do NOT invent information, opinions, problems, praise,
   topics, or recommendations.

3. Combine similar feedback points when appropriate.

4. Keep all outputs concise, specific, and useful for trainers.

5. Do not simply copy student comments word-for-word.

6. Generate the results dynamically based on the feedback
   for THIS {scope_label}.

7. Do not use a fixed list of topics.

8. Return ONLY valid JSON.


SUMMARY
=======

Provide a concise human-readable summary of the overall
student feedback.

The summary should mention:

- Overall response from students
- Important positive feedback
- Important concerns
- Important areas requiring attention

Do not exaggerate the feedback.


WHAT TO CONTINUE
================

List the teaching practices the trainer should KEEP doing
because students clearly responded well to them.

Focus on HOW the trainer taught, not only WHAT topic was taught.

Frame every item as a short action starting with "Continue".

Only include a practice when positive feedback supports it.

Maximum 5 items.

Each item must start with "Continue".


WHAT TO REPEAT
==============

Identify the specific TOPICS or CONCEPTS that students:

- Did not fully understand
- Found difficult
- Requested more practice with
- Requested additional examples for
- Indicated should be explained or revisited again

Output ONLY the topic or concept that needs to be repeated.

Do NOT write a recommendation sentence.

Do NOT describe teaching style.

Do NOT describe trainer behavior.

Do NOT include general improvements such as "slow down".

Do NOT include a topic merely because it was mentioned.

Only include a topic when there is evidence that students
struggled with it, needed more practice, needed more examples,
or needed it revisited.

If there is no topic that clearly needs to be repeated,
return an empty list.

Maximum 5 items.


WHAT TO IMPROVE
===============

Identify actionable improvements related to:

- Teaching pace
- Time allocation
- Doubt clarification
- Practice opportunities
- Explanations
- Visual explanations
- Student support
- Exercises
- Session structure

Only include improvements supported by student feedback.

Do not invent improvements.

Do not simply repeat the topic from "What to Repeat".

Maximum 5 items.


KEY THEMES
==========

Identify the most important subjects, learning activities,
and feedback areas discussed by students.

Themes should be concise labels.

Generate themes dynamically from the current {scope_label}'s feedback.

Do not use a fixed list.

Do not create themes that are not supported by the feedback.

Maximum 6 themes.


OUTPUT FORMAT
=============

Return ONLY valid JSON using exactly this structure:

{{
    "summary": "Concise overall summary of the feedback.",

    "what_to_continue": [
        "Continue using a specific praised teaching practice",
        "Continue providing another praised teaching practice"
    ],

    "what_to_repeat": [
        "Topic that students need to revisit",
        "Another topic that needs revision"
    ],

    "what_to_improve": [
        "Specific training improvement",
        "Another training improvement"
    ],

    "key_themes": [
        "Theme 1",
        "Theme 2",
        "Theme 3"
    ]
}}
"""

    # ---------------------------------------------------------
    # Ollama request
    # ---------------------------------------------------------

    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "format": "json"
    }

    try:

        response = requests.post(
            OLLAMA_URL,
            json=payload,
            timeout=120
        )

        response.raise_for_status()

        result = response.json()

        raw_response = result.get(
            "response",
            ""
        ).strip()

        if not raw_response:
            raise ValueError(
                "Empty response received from Ollama."
            )

        insights = json.loads(raw_response)

        # -----------------------------------------------------
        # Validate list fields
        # -----------------------------------------------------

        what_to_continue = insights.get(
            "what_to_continue",
            []
        )

        what_to_repeat = insights.get(
            "what_to_repeat",
            []
        )

        what_to_improve = insights.get(
            "what_to_improve",
            []
        )

        key_themes = insights.get(
            "key_themes",
            []
        )

        # Ensure lists
        if not isinstance(what_to_continue, list):
            what_to_continue = []

        if not isinstance(what_to_repeat, list):
            what_to_repeat = []

        if not isinstance(what_to_improve, list):
            what_to_improve = []

        if not isinstance(key_themes, list):
            key_themes = []

        # Keep Continue items action-framed
        what_to_continue = [
            item
            if str(item).strip().lower().startswith("continue")
            else f"Continue {str(item).strip().lstrip('- ').lower()}"
            for item in what_to_continue
            if item and str(item).strip()
        ][:5]

        what_to_repeat = [
            str(item).strip()
            for item in what_to_repeat
            if item and str(item).strip()
        ][:5]

        what_to_improve = [
            str(item).strip()
            for item in what_to_improve
            if item and str(item).strip()
        ][:5]

        key_themes = [
            str(item).strip()
            for item in key_themes
            if item and str(item).strip()
        ][:6]

        # -----------------------------------------------------
        # Return
        # -----------------------------------------------------

        return {
            "summary": insights.get(
                "summary",
                "No summary available."
            ),
            "what_to_continue": what_to_continue,
            "what_to_repeat": what_to_repeat,
            "what_to_improve": what_to_improve,
            "key_themes": key_themes
        }

    except requests.exceptions.RequestException as e:

        print(f"LLM Request Error: {e}")

        return {
            "summary": "AI analysis could not be generated.",
            "what_to_continue": [],
            "what_to_repeat": [],
            "what_to_improve": [],
            "key_themes": []
        }

    except json.JSONDecodeError as e:

        print(f"LLM JSON Parsing Error: {e}")

        return {
            "summary": "AI returned an invalid analysis format.",
            "what_to_continue": [],
            "what_to_repeat": [],
            "what_to_improve": [],
            "key_themes": []
        }

    except Exception as e:

        print(f"LLM Error: {e}")

        return {
            "summary": "AI analysis could not be generated.",
            "what_to_continue": [],
            "what_to_repeat": [],
            "what_to_improve": [],
            "key_themes": []
        }

In [102]:
print("OLLAMA_URL:", OLLAMA_URL)
print("OLLAMA_MODEL:", OLLAMA_MODEL)

OLLAMA_URL: http://localhost:11434/api/generate
OLLAMA_MODEL: gpt-oss:120b-cloud


In [103]:
response = requests.get(
    "http://localhost:11434/api/tags",
    timeout=10
)

print(response.status_code)
print(response.text[:500])

200
{"models":[{"name":"gpt-oss:120b-cloud","model":"gpt-oss:120b-cloud","remote_model":"gpt-oss:120b","remote_host":"https://ollama.com:443","modified_at":"2026-07-06T14:06:44.2038535+05:30","size":384,"digest":"569662207105c69bb0eca2f79a3fdf8691ad6301def477a5ec66f8e8bae237e3","details":{"parent_model":"","format":"","family":"gptoss","families":["gptoss"],"parameter_size":"116.8B","quantization_level":"MXFP4","context_length":131072,"embedding_length":2880},"capabilities":["completion","tools","th


In [105]:


MODULE_ID = 1

MODULE_NAME = modules_df.loc[
    modules_df["id"] == MODULE_ID,
    "name"
].iloc[0]

module_feedback_df = pd.read_sql_query(
    """
    SELECT
        f.id AS feedback_id,
        f.student_id,
        f.day_id,
        d.day_number,
        d.title AS day_title,
        f.star_rating,
        f.sentiment_label,
        f.sentiment_score,
        f.remarks
    FROM feedback f
    JOIN days d
        ON f.day_id = d.id
    WHERE d.module_id = ?
      AND f.remarks IS NOT NULL
      AND TRIM(f.remarks) != ''
    ORDER BY d.day_number, f.id
    """,
    conn,
    params=(MODULE_ID,)
)

display(module_feedback_df)

,feedback_id,student_id,day_id,day_number,day_title,star_rating,sentiment_label,sentiment_score,remarks
0,1,1,1,1,Program Onboarding & Digital Literacy,5,Positive,0.1300,The trainer explained the concepts very clearly.
1,3,3,1,1,Program Onboarding & Digital Literacy,5,Positive,0.5500,The trainer was excellent and explained everyt...
2,4,4,1,1,Program Onboarding & Digital Literacy,2,Negative,-0.4000,The explanation was confusing and the session ...
3,5,5,1,1,Program Onboarding & Digital Literacy,3,Neutral,0.0000,The session was conducted today.
4,6,1,1,1,Program Onboarding & Digital Literacy,5,Positive,0.1300,The trainer explained computer fundamentals ve...
5,7,2,2,2,Operating System Basics,4,Positive,0.4333,The operating system concepts were easy to und...
6,39,5,2,2,Operating System Basics,5,Positive,0.5167,I liked the step-by-step OS demonstration on t...
7,8,3,3,3,File Systems & File Management,3,Positive,0.1042,"File management was useful, but the session wa..."
8,9,4,4,4,Computer Hardware Basics,4,Positive,0.7000,The trainer explained computer hardware compon...
9,10,5,5,5,Internet & Networking Basics,2,Positive,0.1000,The networking explanation was confusing and n...


In [106]:
feedback_data = [
    {
        "feedback_id": row["feedback_id"],
        "student_id": row["student_id"],
        "day_id": row["day_id"],
        "day_number": row["day_number"],
        "day_title": row["day_title"],
        "rating": row["star_rating"],
        "sentiment": row["sentiment_label"],
        "sentiment_score": row["sentiment_score"],
        "remarks": row["remarks"]
    }
    for _, row in module_feedback_df.iterrows()
]

In [107]:
print("Number of feedback records:", len(feedback_data))
print(feedback_data[0])

Number of feedback records: 13
{'feedback_id': 1, 'student_id': 1, 'day_id': 1, 'day_number': 1, 'day_title': 'Program Onboarding & Digital Literacy', 'rating': 5, 'sentiment': 'Positive', 'sentiment_score': 0.13, 'remarks': 'The trainer explained the concepts very clearly.'}


In [109]:
module_insights = generate_feedback_insights(
    "module",
    MODULE_NAME,
    feedback_data
)

In [110]:
print("============================================")
print("AI MODULE INSIGHTS")
print("============================================")

print("\nSUMMARY:")
print(module_insights["summary"])

print("\nWHAT TO CONTINUE:")
for item in module_insights["what_to_continue"]:
    print("-", item)

print("\nWHAT TO REPEAT:")
for item in module_insights["what_to_repeat"]:
    print("-", item)

print("\nWHAT TO IMPROVE:")
for item in module_insights["what_to_improve"]:
    print("-", item)

print("\nKEY THEMES:")
for item in module_insights["key_themes"]:
    print("-", item)

AI MODULE INSIGHTS

SUMMARY:
Overall, students responded positively, praising clear and patient explanations, practical step‑by‑step demos, real‑life examples, and helpful review quizzes. A few students found certain sections confusing or too fast, especially networking, password/network security, and file management, indicating a need for clearer, slower coverage of those topics.

WHAT TO CONTINUE:
- Continue explaining concepts clearly and patiently.
- Continue using step‑by‑step practical demonstrations.
- Continue incorporating real‑life examples to illustrate topics.
- Continue providing good examples for hardware components.
- Continue ending modules with short review quizzes.

WHAT TO REPEAT:
- Networking concepts
- Password security
- Network security
- File management

WHAT TO IMPROVE:
- Slow down the pace during fast sections, especially file management.
- Add more practical demonstrations for networking topics.
- Provide simpler, concrete examples for password and network se

#sentiment + topic modeling + LLM insights  helper functions

In [111]:
def _empty_sentiment_distribution():
    return {
        "Positive": {"count": 0, "percentage": 0},
        "Neutral": {"count": 0, "percentage": 0},
        "Negative": {"count": 0, "percentage": 0}
    }


def _prepare_feedback_data(feedback_df):
    """
    Convert a pandas DataFrame into the feedback structure
    expected by the LLM insight function.
    """

    feedback_data = []

    for _, row in feedback_df.iterrows():

        feedback_data.append({
            "feedback_id": int(row["feedback_id"]),
            "student_id": int(row["student_id"]),
            "day_id": int(row["day_id"]),
            "day_number": int(row["day_number"]),
            "day_title": row["day_title"],
            "rating": row["star_rating"],
            "sentiment": row["sentiment_label"],
            "sentiment_score": row["sentiment_score"],
            "remarks": row["remarks"]
        })

    return feedback_data


def _compute_metrics(feedback_data):
    """
    Calculate module-level feedback metrics.
    """

    feedback_count = len(feedback_data)

    unique_students = len(
        set(
            item["student_id"]
            for item in feedback_data
            if item.get("student_id") is not None
        )
    )

    ratings = [
        item["rating"]
        for item in feedback_data
        if item.get("rating") is not None
    ]

    average_rating = (
        round(sum(ratings) / len(ratings), 2)
        if ratings
        else None
    )

    sentiment_counts = {
        "Positive": 0,
        "Neutral": 0,
        "Negative": 0
    }

    sentiment_scores = []

    for item in feedback_data:

        sentiment = item.get("sentiment")

        if sentiment in sentiment_counts:
            sentiment_counts[sentiment] += 1

        score = item.get("sentiment_score")

        if score is not None:
            sentiment_scores.append(score)

    average_sentiment_score = (
        round(
            sum(sentiment_scores) / len(sentiment_scores),
            4
        )
        if sentiment_scores
        else 0
    )

    sentiment_distribution = {}

    for sentiment, count in sentiment_counts.items():

        percentage = (
            round((count / feedback_count) * 100, 1)
            if feedback_count
            else 0
        )

        sentiment_distribution[sentiment] = {
            "count": count,
            "percentage": percentage
        }

    return {
        "students_responded": unique_students,
        "feedback_responses": feedback_count,
        "average_rating": average_rating,
        "average_sentiment_score": average_sentiment_score,
        "sentiment_distribution": sentiment_distribution
    }

#complete module analysis

In [140]:
def get_module_feedback_analysis(
    module_id,
    module_name,
    number_of_topics=3,
    number_of_keywords=5
):
    """
    Generate complete feedback analysis for a module.

    Includes:
    - Feedback records
    - Average rating
    - Sentiment metrics
    - LDA topic modeling
    - LLM-generated topic names
    - LLM-generated feedback insights
    """

    # -----------------------------------------
    # 1. Get module feedback from SQLite
    # -----------------------------------------

    module_feedback_df = pd.read_sql_query(
        """
        SELECT
            f.id AS feedback_id,
            f.student_id,
            f.day_id,
            d.day_number,
            d.title AS day_title,
            f.star_rating,
            f.sentiment_label,
            f.sentiment_score,
            f.remarks
        FROM feedback f
        JOIN days d
            ON f.day_id = d.id
        WHERE d.module_id = ?
          AND f.remarks IS NOT NULL
          AND TRIM(f.remarks) != ''
        ORDER BY d.day_number, f.id
        """,
        conn,
        params=(module_id,)
    )

    # -----------------------------------------
    # 2. Empty result
    # -----------------------------------------

    if module_feedback_df.empty:

        return {
            "scope_type": "module",
            "scope_id": module_id,
            "scope_name": module_name,
            "status": "NO_DATA",

            "module_id": module_id,
            "module_name": module_name,

            "students_responded": 0,
            "feedback_responses": 0,

            "average_rating": None,
            "average_sentiment_score": 0,

            "sentiment_distribution":
                _empty_sentiment_distribution(),

            "topics": [],

            "summary":
                "No feedback is available for this module yet.",

            "what_to_continue": [],
            "what_to_repeat": [],
            "what_to_improve": [],
            "key_themes": [],

            "feedback": []
        }

    # -----------------------------------------
    # 3. Prepare feedback data
    # -----------------------------------------

    feedback_data = _prepare_feedback_data(
        module_feedback_df
    )

    # -----------------------------------------
    # 4. Calculate metrics
    # -----------------------------------------

    metrics = _compute_metrics(
        feedback_data
    )

    # -----------------------------------------
    # 5. Topic modeling
    # -----------------------------------------

    module_documents = (
        module_feedback_df["remarks"]
        .dropna()
        .astype(str)
        .tolist()
    )

    try:

        # LDA discovers topics
        topics = extract_topics(
            module_documents,
            number_of_topics=number_of_topics,
            number_of_keywords=number_of_keywords
        )

        # LLM generates human-readable topic names
        topics = add_topic_names(topics)

    except Exception as e:

        print(
            f"Topic modeling error: {e}"
        )

        topics = []

    # -----------------------------------------
    # 6. Generate LLM insights
    # -----------------------------------------

    try:

        insights = generate_feedback_insights(
            "module",
            module_name,
            feedback_data
        )

    except Exception as e:

        print(
            f"LLM insight error: {e}"
        )

        insights = {
            "summary":
                "Unable to generate AI insights.",

            "what_to_continue": [],

            "what_to_repeat": [],

            "what_to_improve": [],

            "key_themes": []
        }

    # -----------------------------------------
    # 7. Build final result
    # -----------------------------------------

    result = {

        "scope_type": "module",

        "scope_id": module_id,

        "scope_name": module_name,

        "status": "COMPLETED",

        "module_id": module_id,

        "module_name": module_name,

        # -------------------------------------
        # Metrics
        # -------------------------------------

        "students_responded":
            metrics["students_responded"],

        "feedback_responses":
            metrics["feedback_responses"],

        "average_rating":
            metrics["average_rating"],

        "average_sentiment_score":
            metrics["average_sentiment_score"],

        "sentiment_distribution":
            metrics["sentiment_distribution"],

        # -------------------------------------
        # Topics
        # -------------------------------------

        "topics": topics,

        # -------------------------------------
        # AI Insights
        # -------------------------------------

        "summary":
            insights.get(
                "summary",
                "No summary available."
            ),

        "what_to_continue":
            insights.get(
                "what_to_continue",
                []
            ),

        "what_to_repeat":
            insights.get(
                "what_to_repeat",
                []
            ),

        "what_to_improve":
            insights.get(
                "what_to_improve",
                []
            ),

        "key_themes":
            insights.get(
                "key_themes",
                []
            ),

        # -------------------------------------
        # Raw feedback
        # -------------------------------------

        "feedback": feedback_data
    }

    return result

In [141]:
MODULE_ID = 1

MODULE_NAME = modules_df.loc[
    modules_df["id"] == MODULE_ID,
    "name"
].iloc[0]

module_analysis = get_module_feedback_analysis(
    module_id=MODULE_ID,
    module_name=MODULE_NAME,
    number_of_topics=3,
    number_of_keywords=5
)

In [142]:
for topic in module_analysis["topics"]:

    print(
        f"Topic {topic['topic_number']}: "
        f"{topic['topic_name']}"
    )

    print(
        "Keywords:",
        ", ".join(topic["keywords"])
    )

    print(
        "Weights:",
        topic["topic_weights"]
    )

    print(
        "Document Count:",
        topic["document_count"]
    )

    print()

Topic 1: File System Operations
Keywords: step, file, os, liked, operations
Weights: [0.3041, 0.1746, 0.1738, 0.1738, 0.1738]
Document Count: 13

Topic 2: Real-Life Cybersecurity Concepts
Keywords: concepts, cybersecurity, phishing, life, real
Weights: [0.304, 0.1745, 0.1739, 0.1739, 0.1739]
Document Count: 13

Topic 3: Computer Fundamentals Review
Keywords: review, fundamentals, confusing, using, computer
Weights: [0.2414, 0.2413, 0.2411, 0.1381, 0.138]
Document Count: 13



In [143]:
print("============================================")
print("COMPLETE MODULE ANALYSIS")
print("============================================")

print("\nMODULE:")
print(module_analysis["module_name"])

print("\nSTATUS:")
print(module_analysis["status"])

print("\nSTUDENTS RESPONDED:")
print(module_analysis["students_responded"])

print("\nFEEDBACK RESPONSES:")
print(module_analysis["feedback_responses"])

print("\nAVERAGE RATING:")
print(module_analysis["average_rating"])

print("\nAVERAGE SENTIMENT SCORE:")
print(module_analysis["average_sentiment_score"])

print("\nSENTIMENT DISTRIBUTION:")
print(module_analysis["sentiment_distribution"])

COMPLETE MODULE ANALYSIS

MODULE:
Onboarding & Computer Fundamentals

STATUS:
COMPLETED

STUDENTS RESPONDED:
5

FEEDBACK RESPONSES:
13

AVERAGE RATING:
3.85

AVERAGE SENTIMENT SCORE:
0.2139

SENTIMENT DISTRIBUTION:
{'Positive': {'count': 10, 'percentage': 76.9}, 'Neutral': {'count': 2, 'percentage': 15.4}, 'Negative': {'count': 1, 'percentage': 7.7}}


In [144]:
print("\n============================================")
print("TOPICS")
print("============================================")

for topic in module_analysis["topics"]:

    print(
        f"\nTopic {topic['topic_number']}: "
        f"{topic['topic_name']}"
    )

    print(
        "Keywords:",
        ", ".join(topic["keywords"])
    )

    print(
        "Document Count:",
        topic["document_count"]
    )

    print(
        "Weights:",
        topic["topic_weights"]
    )


TOPICS

Topic 1: File System Operations
Keywords: step, file, os, liked, operations
Document Count: 13
Weights: [0.3041, 0.1746, 0.1738, 0.1738, 0.1738]

Topic 2: Real-Life Cybersecurity Concepts
Keywords: concepts, cybersecurity, phishing, life, real
Document Count: 13
Weights: [0.304, 0.1745, 0.1739, 0.1739, 0.1739]

Topic 3: Computer Fundamentals Review
Keywords: review, fundamentals, confusing, using, computer
Document Count: 13
Weights: [0.2414, 0.2413, 0.2411, 0.1381, 0.138]


In [116]:
print("\n============================================")
print("AI MODULE INSIGHTS")
print("============================================")

print("\nSUMMARY:")
print(module_analysis["summary"])

print("\nWHAT TO CONTINUE:")

for item in module_analysis["what_to_continue"]:
    print("-", item)

print("\nWHAT TO REPEAT:")

for item in module_analysis["what_to_repeat"]:
    print("-", item)

print("\nWHAT TO IMPROVE:")

for item in module_analysis["what_to_improve"]:
    print("-", item)

print("\nKEY THEMES:")

for item in module_analysis["key_themes"]:
    print("-", item)


AI MODULE INSIGHTS

SUMMARY:
Overall feedback is largely positive, highlighting clear explanations, step‑by‑step demos, practical examples and helpful quizzes. Concerns include confusion around networking and password/network security, and a pace that felt too fast for some topics.

WHAT TO CONTINUE:
- Continue delivering clear and patient explanations of concepts
- Continue using step‑by‑step on‑screen demonstrations for operating system tasks
- Continue incorporating practical, real‑life examples such as phishing scenarios for cybersecurity
- Continue providing good examples for computer hardware components
- Continue ending modules with short review quizzes

WHAT TO REPEAT:
- Networking
- Password security
- Network security
- File Management

WHAT TO IMPROVE:
- Slow down the overall session pace
- Allocate more time for topics that students found confusing
- Add additional practical demonstrations for networking concepts
- Provide simpler examples for password and network security

In [145]:
print("\n============================================")
print("RAW FEEDBACK")
print("============================================")

for item in module_analysis["feedback"]:

    print(
        f"\nFeedback ID: {item['feedback_id']}"
    )

    print(
        f"Day {item['day_number']}: "
        f"{item['day_title']}"
    )

    print(
        f"Rating: {item['rating']}"
    )

    print(
        f"Sentiment: {item['sentiment']}"
    )

    print(
        f"Remarks: {item['remarks']}"
    )


RAW FEEDBACK

Feedback ID: 1
Day 1: Program Onboarding & Digital Literacy
Rating: 5
Sentiment: Positive
Remarks: The trainer explained the concepts very clearly.

Feedback ID: 3
Day 1: Program Onboarding & Digital Literacy
Rating: 5
Sentiment: Positive
Remarks: The trainer was excellent and explained everything clearly.

Feedback ID: 4
Day 1: Program Onboarding & Digital Literacy
Rating: 2
Sentiment: Negative
Remarks: The explanation was confusing and the session was difficult to understand.

Feedback ID: 5
Day 1: Program Onboarding & Digital Literacy
Rating: 3
Sentiment: Neutral
Remarks: The session was conducted today.

Feedback ID: 6
Day 1: Program Onboarding & Digital Literacy
Rating: 5
Sentiment: Positive
Remarks: The trainer explained computer fundamentals very clearly and patiently.

Feedback ID: 7
Day 2: Operating System Basics
Rating: 4
Sentiment: Positive
Remarks: The operating system concepts were easy to understand with practical examples.

Feedback ID: 39
Day 2: Operating

In [118]:
module_analysis.keys()

dict_keys(['scope_type', 'scope_id', 'scope_name', 'status', 'module_id', 'module_name', 'students_responded', 'feedback_responses', 'average_rating', 'average_sentiment_score', 'sentiment_distribution', 'topics', 'summary', 'what_to_continue', 'what_to_repeat', 'what_to_improve', 'key_themes', 'feedback'])

In [146]:
result = get_module_feedback_analysis(
    module_id=1,
    module_name="Onboarding & Computer Fundamentals",
    number_of_topics=3,
    number_of_keywords=5
)

print("Module:", result["module_name"])
print("Feedback Responses:", result["feedback_responses"])
print("Average Rating:", result["average_rating"])
print("Average Sentiment:", result["average_sentiment_score"])

print("\nSentiment Distribution:")
print(result["sentiment_distribution"])

print("\nTopics:")
for topic in result["topics"]:
    print(f"\nTopic {topic['topic_number']}: {topic['topic_name']}")
    print("Keywords:", ", ".join(topic["keywords"]))
    print("Documents:", topic["document_count"])

print("\nSummary:")
print(result["summary"])

print("\nWhat to Continue:")
for item in result["what_to_continue"]:
    print("-", item)

print("\nWhat to Repeat:")
for item in result["what_to_repeat"]:
    print("-", item)

print("\nWhat to Improve:")
for item in result["what_to_improve"]:
    print("-", item)

print("\nKey Themes:")
for item in result["key_themes"]:
    print("-", item)

Module: Onboarding & Computer Fundamentals
Feedback Responses: 13
Average Rating: 3.85
Average Sentiment: 0.2139

Sentiment Distribution:
{'Positive': {'count': 10, 'percentage': 76.9}, 'Neutral': {'count': 2, 'percentage': 15.4}, 'Negative': {'count': 1, 'percentage': 7.7}}

Topics:

Topic 1: OS File Operations
Keywords: step, file, os, liked, operations
Documents: 13

Topic 2: Real-World Cybersecurity Concepts
Keywords: concepts, cybersecurity, phishing, life, real
Documents: 13

Topic 3: Computer Fundamentals Review
Keywords: review, fundamentals, confusing, using, computer
Documents: 13

Summary:
Most students responded positively, praising the trainer's clear, patient explanations and step‑by‑step practical demonstrations, especially in OS and cybersecurity scenarios. However, a few students found certain parts confusing—particularly networking and some cybersecurity concepts—and noted that the pace was a bit fast for file management.

What to Continue:
- Continue explaining conce

In [147]:
modules_df = pd.read_sql_query(
    """
    SELECT
        id AS module_id,
        module_number,
        name AS module_name
    FROM modules
    ORDER BY module_number
    """,
    conn
)

modules_df

,module_id,module_number,module_name
0,1,1,Onboarding & Computer Fundamentals
1,2,2,"HTML5, CSS3, Flexbox/Grid"
2,3,3,Frontend Development
3,4,4,Python Programming I
4,5,5,Python II & Git/GitHub
5,6,6,Data Structures & Algorithms I


module summary table

In [149]:
import pandas as pd

module_summary = []

for result in all_module_results:

    sentiment = result["sentiment_distribution"]

    module_summary.append({
        "Module": result["module_name"],
        "Feedback": result["feedback_responses"],
        "Avg Rating": result["average_rating"],
        "Avg Sentiment": result["average_sentiment_score"],

        "Positive %":
            sentiment["Positive"]["percentage"],

        "Neutral %":
            sentiment["Neutral"]["percentage"],

        "Negative %":
            sentiment["Negative"]["percentage"],

        "What to Improve":
            "; ".join(result["what_to_improve"]),

        "What to Repeat":
            "; ".join(result["what_to_repeat"]),

        "Key Themes":
            "; ".join(result["key_themes"])
    })

module_summary_df = pd.DataFrame(module_summary)

module_summary_df

,Module,Feedback,Avg Rating,Avg Sentiment,Positive %,Neutral %,Negative %,What to Improve,What to Repeat,Key Themes
0,Onboarding & Computer Fundamentals,13,3.85,0.2139,76.9,15.4,7.7,Slow down the overall session pace to allow be...,Networking concepts; Password and network secu...,Clear explanations; Step‑by‑step demonstration...
1,"HTML5, CSS3, Flexbox/Grid",15,3.87,0.1934,86.7,0.0,13.3,Allocate more time for hands‑on exercises so s...,Flexbox nested layouts; Absolute and relative ...,Live coding demonstrations; Practical exercise...
2,Frontend Development,13,4.23,0.2759,76.9,7.7,15.4,Slow down the pacing when covering complex top...,Fetch API and JSON handling; Debugging workflo...,Clear examples & real‑world datasets; Hands‑on...
3,Python Programming I,13,4.00,0.1786,69.2,23.1,7.7,Extend guided practice time for for‑ and while...,Nested loops; Nested dictionaries; Functions,Clear explanations with simple examples; Inter...
4,Python II & Git/GitHub,11,3.82,0.2728,81.8,18.2,0.0,Slow the pace of introductory Git commands to ...,Git add/commit/status; Pushing code to GitHub;...,Advanced Python tasks; CSV file handling; Git ...
5,Data Structures & Algorithms I,11,3.82,0.1367,45.5,27.3,27.3,Incorporate more visual explanations and diagr...,Recursion; Bubble sort vs insertion sort compa...,Practical examples & analogies; Visual aids & ...


In [150]:
all_module_results = []

for _, module in modules_df.iterrows():

    print("=" * 70)
    print(f"Analyzing Module {module['module_number']}: {module['module_name']}")
    print("=" * 70)

    result = get_module_feedback_analysis(
        module_id=int(module["module_id"]),
        module_name=module["module_name"],
        number_of_topics=3,
        number_of_keywords=5
    )

    all_module_results.append(result)

    print("Feedback:", result["feedback_responses"])
    print("Average Rating:", result["average_rating"])
    print("Average Sentiment:", result["average_sentiment_score"])

    print("\nTopics:")
    for topic in result["topics"]:
        print(
            f"  {topic['topic_number']}: "
            f"{topic['topic_name']}"
        )

    print()

Analyzing Module 1: Onboarding & Computer Fundamentals
Feedback: 13
Average Rating: 3.85
Average Sentiment: 0.2139

Topics:
  1: Operating System File Operations
  2: Real-Life Cybersecurity Concepts
  3: Computer Fundamentals Review

Analyzing Module 2: HTML5, CSS3, Flexbox/Grid
Feedback: 15
Average Rating: 3.87
Average Sentiment: 0.1934

Topics:
  1: Web Page Navigation
  2: Live Responsive Flexbox Layouts
  3: CSS Positioning Performance

Analyzing Module 3: Frontend Development
Feedback: 13
Average Rating: 4.23
Average Sentiment: 0.2759

Topics:
  1: JavaScript API Data Handling
  2: DOM Demo Validation
  3: Responsive Debugging Concepts

Analyzing Module 4: Python Programming I
Feedback: 13
Average Rating: 4.0
Average Sentiment: 0.1786

Topics:
  1: Python Loop Practice
  2: Nested Dictionaries in Python
  3: Python Mini Table Handling

Analyzing Module 5: Python II & Git/GitHub
Feedback: 11
Average Rating: 3.82
Average Sentiment: 0.2728

Topics:
  1: Remote Git Repository Managem

In [151]:
import pandas as pd

module_summary = []

for result in all_module_results:

    sentiment = result["sentiment_distribution"]

    module_summary.append({
        "Module": result["module_name"],
        "Feedback": result["feedback_responses"],
        "Avg Rating": result["average_rating"],
        "Avg Sentiment": result["average_sentiment_score"],

        "Positive %":
            sentiment["Positive"]["percentage"],

        "Neutral %":
            sentiment["Neutral"]["percentage"],

        "Negative %":
            sentiment["Negative"]["percentage"],

        "What to Improve":
            "; ".join(result["what_to_improve"]),

        "What to Repeat":
            "; ".join(result["what_to_repeat"]),

        "Key Themes":
            "; ".join(result["key_themes"])
    })

module_summary_df = pd.DataFrame(module_summary)

module_summary_df

,Module,Feedback,Avg Rating,Avg Sentiment,Positive %,Neutral %,Negative %,What to Improve,What to Repeat,Key Themes
0,Onboarding & Computer Fundamentals,13,3.85,0.2139,76.9,15.4,7.7,Slow down the pacing for faster topics; Add mo...,Networking concepts; Password and network secu...,Clear explanations; Step‑by‑step demonstration...
1,"HTML5, CSS3, Flexbox/Grid",15,3.87,0.1934,86.7,0.0,13.3,Slow down the teaching pace to allow better co...,Absolute and relative positioning; Flexbox nes...,Responsive design; Hands‑on practical exercise...
2,Frontend Development,13,4.23,0.2759,76.9,7.7,15.4,"Slow down the overall session pace, especially...",JavaScript events; Callback functions in event...,Core JavaScript fundamentals; Hands‑on DOM man...
3,Python Programming I,13,4.00,0.1786,69.2,23.1,7.7,Allocate more time for doubt clarification dur...,Loops (including nested for/while loops); Func...,Clear presentation of basics; Hands‑on setup a...
4,Python II & Git/GitHub,11,3.82,0.2728,81.8,18.2,0.0,Slow down the pace of introductory Git concept...,Git add/commit/status; Merge conflict resoluti...,Advanced Python practice; File handling with C...
5,Data Structures & Algorithms I,11,3.82,0.1367,45.5,27.3,27.3,Increase visual explanations and diagrams for ...,Recursion; Bubble sort vs Insertion sort compa...,Practical examples; Visual diagrams; Algorithm...


Day-wise analysis.

In [152]:
def get_day_feedback_analysis(
    day_id,
    day_title,
    day_number,
    module_name,
    number_of_topics=3,
    number_of_keywords=5
):
    """
    Generate complete feedback analysis for a single day.

    Includes:
    - Feedback records
    - Average rating
    - Sentiment metrics
    - LDA topic modeling
    - LLM-generated topic names
    - LLM-generated insights
    """

    # -----------------------------------------
    # 1. Get day feedback from SQLite
    # -----------------------------------------

    day_feedback_df = pd.read_sql_query(
        """
        SELECT
            f.id AS feedback_id,
            f.student_id,
            f.day_id,
            d.day_number,
            d.title AS day_title,
            f.star_rating,
            f.sentiment_label,
            f.sentiment_score,
            f.remarks
        FROM feedback f
        JOIN days d
            ON f.day_id = d.id
        WHERE f.day_id = ?
          AND f.remarks IS NOT NULL
          AND TRIM(f.remarks) != ''
        ORDER BY f.id
        """,
        conn,
        params=(day_id,)
    )

    # -----------------------------------------
    # 2. Empty result
    # -----------------------------------------

    scope_name = (
        f"Day {day_number}: {day_title}"
        if day_title
        else f"Day {day_number}"
    )

    if day_feedback_df.empty:

        return {
            "scope_type": "day",
            "scope_id": day_id,
            "scope_name": scope_name,
            "status": "NO_DATA",

            "day_id": day_id,
            "day_number": day_number,
            "day_title": day_title,
            "module_name": module_name,

            "students_responded": 0,
            "feedback_responses": 0,

            "average_rating": None,
            "average_sentiment_score": 0,

            "sentiment_distribution":
                _empty_sentiment_distribution(),

            "topics": [],

            "summary":
                "No feedback is available for this day yet.",

            "what_to_continue": [],
            "what_to_repeat": [],
            "what_to_improve": [],
            "key_themes": [],

            "feedback": []
        }

    # -----------------------------------------
    # 3. Prepare feedback data
    # -----------------------------------------

    feedback_data = _prepare_feedback_data(
        day_feedback_df
    )

    # -----------------------------------------
    # 4. Calculate metrics
    # -----------------------------------------

    metrics = _compute_metrics(
        feedback_data
    )

    # -----------------------------------------
    # 5. Topic modeling
    # -----------------------------------------

    day_documents = (
        day_feedback_df["remarks"]
        .dropna()
        .astype(str)
        .tolist()
    )

    try:

        topics = extract_topics(
            day_documents,
            number_of_topics=number_of_topics,
            number_of_keywords=number_of_keywords
        )

        # Generate human-readable topic names
        topics = add_topic_names(topics)

    except Exception as e:

        print(
            f"Topic modeling error: {e}"
        )

        topics = []

    # -----------------------------------------
    # 6. Generate LLM insights
    # -----------------------------------------

    try:

        insights = generate_feedback_insights(
            "day",
            scope_name,
            feedback_data
        )

    except Exception as e:

        print(
            f"LLM insight error: {e}"
        )

        insights = {
            "summary":
                "Unable to generate AI insights.",

            "what_to_continue": [],
            "what_to_repeat": [],
            "what_to_improve": [],
            "key_themes": []
        }

    # -----------------------------------------
    # 7. Build final result
    # -----------------------------------------

    result = {

        "scope_type": "day",

        "scope_id": day_id,

        "scope_name": scope_name,

        "status": "COMPLETED",

        "day_id": day_id,

        "day_number": day_number,

        "day_title": day_title,

        "module_name": module_name,

        # Metrics
        "students_responded":
            metrics["students_responded"],

        "feedback_responses":
            metrics["feedback_responses"],

        "average_rating":
            metrics["average_rating"],

        "average_sentiment_score":
            metrics["average_sentiment_score"],

        "sentiment_distribution":
            metrics["sentiment_distribution"],

        # Topics
        "topics": topics,

        # AI Insights
        "summary":
            insights.get(
                "summary",
                "No summary available."
            ),

        "what_to_continue":
            insights.get(
                "what_to_continue",
                []
            ),

        "what_to_repeat":
            insights.get(
                "what_to_repeat",
                []
            ),

        "what_to_improve":
            insights.get(
                "what_to_improve",
                []
            ),

        "key_themes":
            insights.get(
                "key_themes",
                []
            ),

        # Raw feedback
        "feedback": feedback_data
    }

    return result

In [155]:
days_df = pd.read_sql_query(
    """
    SELECT
        d.id AS day_id,
        d.module_id AS module_id,
        m.module_number AS module_number,
        m.name AS module_name,
        d.day_number AS day_number,
        d.title AS day_title
    FROM days d
    JOIN modules m
        ON d.module_id = m.id
    ORDER BY
        m.module_number,
        d.day_number
    """,
    conn
)

print(days_df.columns.tolist())
print(days_df.head(10))

['day_id', 'module_id', 'module_number', 'module_name', 'day_number', 'day_title']
   day_id  module_id  module_number                         module_name  \
0       1          1              1  Onboarding & Computer Fundamentals   
1       2          1              1  Onboarding & Computer Fundamentals   
2       3          1              1  Onboarding & Computer Fundamentals   
3       4          1              1  Onboarding & Computer Fundamentals   
4       5          1              1  Onboarding & Computer Fundamentals   
5       6          1              1  Onboarding & Computer Fundamentals   
6       7          1              1  Onboarding & Computer Fundamentals   
7       8          2              2           HTML5, CSS3, Flexbox/Grid   
8       9          2              2           HTML5, CSS3, Flexbox/Grid   
9      10          2              2           HTML5, CSS3, Flexbox/Grid   

   day_number                              day_title  
0           1  Program Onboarding & 

In [156]:
test_day = days_df.iloc[0]

day_result = get_day_feedback_analysis(
    day_id=int(test_day["day_id"]),
    day_title=test_day["day_title"],
    day_number=int(test_day["day_number"]),
    module_name=test_day["module_name"],
    number_of_topics=3,
    number_of_keywords=5
)

In [157]:
print("Day:", day_result["scope_name"])
print("Module:", day_result["module_name"])

print("\nFeedback Responses:", day_result["feedback_responses"])
print("Average Rating:", day_result["average_rating"])
print("Average Sentiment:", day_result["average_sentiment_score"])

print("\nSentiment Distribution:")
print(day_result["sentiment_distribution"])

print("\nTopics:")
for topic in day_result["topics"]:
    print(
        f"{topic['topic_number']}: "
        f"{topic['topic_name']}"
    )

print("\nSummary:")
print(day_result["summary"])

print("\nWhat to Continue:")
for item in day_result["what_to_continue"]:
    print("-", item)

print("\nWhat to Repeat:")
for item in day_result["what_to_repeat"]:
    print("-", item)

print("\nWhat to Improve:")
for item in day_result["what_to_improve"]:
    print("-", item)

print("\nKey Themes:")
for item in day_result["key_themes"]:
    print("-", item)

Day: Day 1: Program Onboarding & Digital Literacy
Module: Onboarding & Computer Fundamentals

Feedback Responses: 5
Average Rating: 4.0
Average Sentiment: 0.082

Sentiment Distribution:
{'Positive': {'count': 3, 'percentage': 60.0}, 'Neutral': {'count': 1, 'percentage': 20.0}, 'Negative': {'count': 1, 'percentage': 20.0}}

Topics:
1: Excellent Patiently Conducted Confusing Concepts
2: Patiently Conducted Computer Fundamentals
3: Patiently Clarifying Confusing Concepts

Summary:
Most students responded positively, praising the trainer’s clear and patient explanations. One student found the explanation confusing and the session difficult to follow, indicating a need for clearer communication on some concepts.

What to Continue:
- Continue delivering clear and concise explanations of concepts
- Continue explaining material patiently and step‑by‑step
- Continue maintaining an excellent and supportive teaching demeanor

What to Repeat:

What to Improve:
- Improve the clarity of explanations

In [159]:
all_day_results = []

for _, day in days_df.iterrows():

    print("=" * 80)
    print(
        f"Analyzing Module {int(day['module_number'])} "
        f"- Day {int(day['day_number'])}: "
        f"{day['day_title']}"
    )
    print("=" * 80)

    result = get_day_feedback_analysis(
        day_id=int(day["day_id"]),
        day_title=day["day_title"],
        day_number=int(day["day_number"]),
        module_name=day["module_name"],
        number_of_topics=3,
        number_of_keywords=5
    )

    all_day_results.append(result)

    print("Feedback:", result["feedback_responses"])
    print("Average Rating:", result["average_rating"])
    print("Average Sentiment:", result["average_sentiment_score"])

    print("\nSentiment:")
    print(result["sentiment_distribution"])

    print("\nTopics:")
    for topic in result["topics"]:
        print(
            f"  {topic['topic_number']}: "
            f"{topic['topic_name']}"
        )

    print("\nWhat to Improve:")
    for item in result["what_to_improve"]:
        print(f"  • {item}")

    print()

Analyzing Module 1 - Day 1: Program Onboarding & Digital Literacy
Feedback: 5
Average Rating: 4.0
Average Sentiment: 0.082

Sentiment:
{'Positive': {'count': 3, 'percentage': 60.0}, 'Neutral': {'count': 1, 'percentage': 20.0}, 'Negative': {'count': 1, 'percentage': 20.0}}

Topics:
  1: Effective Teaching Practices
  2: Fundamental Computer Concepts
  3: Confusing Concepts Explained

What to Improve:
  • Improve clarity of explanations to reduce confusion
  • Allocate additional time for doubt clarification
  • Adjust pacing to ensure all concepts are understood

Analyzing Module 1 - Day 2: Operating System Basics
Feedback: 2
Average Rating: 4.5
Average Sentiment: 0.475

Sentiment:
{'Positive': {'count': 2, 'percentage': 100.0}, 'Neutral': {'count': 0, 'percentage': 0.0}, 'Negative': {'count': 0, 'percentage': 0.0}}

Topics:
  1: OS Screen Operations Steps
  2: Operating System Basics

What to Improve:

Analyzing Module 1 - Day 3: File Systems & File Management
Feedback: 1
Average Ratin

In [160]:
day_summary = []

for result in all_day_results:

    sentiment = result["sentiment_distribution"]

    day_summary.append({
        "Module": result["module_name"],
        "Day": result["day_number"],
        "Day Title": result["day_title"],
        "Feedback": result["feedback_responses"],
        "Avg Rating": result["average_rating"],
        "Avg Sentiment": result["average_sentiment_score"],

        "Positive %":
            sentiment["Positive"]["percentage"],

        "Neutral %":
            sentiment["Neutral"]["percentage"],

        "Negative %":
            sentiment["Negative"]["percentage"],

        "Topics":
            "; ".join(
                topic["topic_name"]
                for topic in result["topics"]
            ),

        "What to Improve":
            "; ".join(result["what_to_improve"]),

        "What to Repeat":
            "; ".join(result["what_to_repeat"]),

        "Key Themes":
            "; ".join(result["key_themes"])
    })

day_summary_df = pd.DataFrame(day_summary)

day_summary_df

,Module,Day,Day Title,Feedback,Avg Rating,Avg Sentiment,Positive %,Neutral %,Negative %,Topics,What to Improve,What to Repeat,Key Themes
0,Onboarding & Computer Fundamentals,1,Program Onboarding & Digital Literacy,5,4.0,0.0820,60.0,20.0,20.0,Effective Teaching Practices; Fundamental Comp...,Improve clarity of explanations to reduce conf...,,Clear explanations; Confusing explanations; Tr...
1,Onboarding & Computer Fundamentals,2,Operating System Basics,2,4.5,0.4750,100.0,0.0,0.0,OS Screen Operations Steps; Operating System B...,,,Practical examples; Step‑by‑step demonstration...
2,Onboarding & Computer Fundamentals,3,File Systems & File Management,1,3.0,0.1042,100.0,0.0,0.0,Fast Small File Management,Slow down the pacing of the session to give st...,,Useful file management content; Fast pacing
3,Onboarding & Computer Fundamentals,4,Computer Hardware Basics,1,4.0,0.7000,100.0,0.0,0.0,Computer Hardware Components,,,Hardware Component Explanations; Use of Good E...
4,Onboarding & Computer Fundamentals,5,Internet & Networking Basics,1,2.0,0.1000,100.0,0.0,0.0,Confusing Networking Demonstrations,Provide clearer explanations for networking co...,Networking explanation,Confusing networking content; Insufficient pra...
5,Onboarding & Computer Fundamentals,6,Cybersecurity Awareness,2,4.0,0.2583,100.0,0.0,0.0,Real-World Cybersecurity Scenarios; Simple Net...,"Provide simpler, more relatable examples for p...",Password security; Network security,Phishing scenario examples; Engagement; Clarit...
6,Onboarding & Computer Fundamentals,7,Module Review,1,4.0,0.0000,0.0,100.0,0.0,Short Quiz Review,,,Module Review; Quiz-based Revision; Fundamenta...
7,"HTML5, CSS3, Flexbox/Grid",1,HTML Structure & Semantic Tags,1,5.0,0.2000,100.0,0.0,0.0,HTML & CSS Concepts,,,Clear concept explanations; Practical exercise...
8,"HTML5, CSS3, Flexbox/Grid",2,CSS Selectors & Styling Basics,1,5.0,0.4333,100.0,0.0,0.0,Frontend Layout Development,,,Practical examples; Flexbox; Grid; Ease of und...
9,"HTML5, CSS3, Flexbox/Grid",3,Box Model & Spacing,1,4.0,0.3000,100.0,0.0,0.0,CSS Assignment Skills,,,Practical assignments; CSS skill development
